In [ ]:

%pip install -q -U crewai crewai-tools ddgs


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5

In [ ]:

import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it in Colab Secrets."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("Gemini API key loaded successfully.")


Gemini API key loaded successfully.


In [ ]:

import os
import json
from pathlib import Path
from datetime import datetime, timezone

from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process, LLM
from crewai.flow.flow import Flow, start, listen, router, or_
from ddgs import DDGS

llm = LLM(
    model="gemini/gemini-3.1-flash-lite-preview",
    api_key=GEMINI_API_KEY,
    temperature=0.4
)

OUTPUT_DIR = Path("/content/approval_report_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def web_search(query: str, max_results: int = 5) -> str:
    """Search DuckDuckGo for public web pages."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        return "\n\n".join(
            f"Title: {item.get('title', '')}\n"
            f"URL: {item.get('href', '')}\n"
            f"Snippet: {item.get('body', '')}"
            for item in results
        ) or "No search results found."
    except Exception as exc:
        return f"Search unavailable: {exc}"

print("Gemini, DuckDuckGo search, and output directory ready.")


Gemini, DuckDuckGo search, and output directory ready.


In [ ]:

class ApprovalReportState(BaseModel):
    topic: str = ""
    audience: str = ""
    research: str = ""
    draft: str = ""
    feedback: str = ""
    decision: str = ""
    revision_count: int = 0
    max_revisions: int = 2
    final_path: str = ""
    status: str = "pending"

def save_text(filename: str, content: str) -> str:
    path = OUTPUT_DIR / filename
    path.write_text(content, encoding="utf-8")
    return str(path)

def save_json(filename: str, data: dict) -> str:
    path = OUTPUT_DIR / filename
    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    return str(path)

def run_crew(role, goal, backstory, task_description, expected_output):
    agent = Agent(
        role=role,
        goal=goal,
        backstory=backstory,
        llm=llm,
        verbose=False
    )

    task = Task(
        description=task_description,
        expected_output=expected_output,
        agent=agent
    )

    crew = Crew(
        agents=[agent],
        tasks=[task],
        process=Process.sequential,
        verbose=False
    )

    return str(crew.kickoff())

print("State model and helper functions ready.")


State model and helper functions ready.


In [ ]:

class ApprovalReportFlow(Flow[ApprovalReportState]):

    @start()
    def collect_topic(self):
        print("=== Approval Report Flow ===")

        self.state.topic = input("Enter the report topic: ").strip()
        if not self.state.topic:
            raise ValueError("The topic cannot be empty.")

        self.state.audience = input(
            "Enter the target audience (e.g., students, managers): "
        ).strip() or "General audience"

        self.state.max_revisions = 2
        self.state.revision_count = 0
        self.state.status = "pending"

        print(f"\nTopic: {self.state.topic}")
        print(f"Audience: {self.state.audience}")

    @listen(collect_topic)
    def research_topic(self):
        print("\n=== Researching Topic ===")

        search_results = web_search(
            f"{self.state.topic} key facts recent developments",
            max_results=5
        )

        if not search_results.strip() or "No search results found" in search_results:
            search_results = "No usable search results were found. Clearly identify information gaps."

        research_task = f"""
        Research the following topic for a report.

        Topic: {self.state.topic}
        Target audience: {self.state.audience}

        Available web search results:
        {search_results}

        Instructions:
        - Summarize the most relevant findings.
        - Identify important facts, trends, and implications.
        - Use only information supported by the supplied results.
        - Include relevant source URLs where available.
        - Clearly identify gaps instead of inventing facts.
        """

        self.state.research = run_crew(
            role="Research Analyst",
            goal="Prepare an evidence-based research brief.",
            backstory="You carefully analyze information and distinguish evidence from uncertainty.",
            task_description=research_task,
            expected_output="A concise research brief with findings and source URLs."
        )

        print("Research completed.")

    @listen(research_topic)
    def write_report(self):
        print("\n=== Drafting Report ===")

        draft_task = f"""
        Write a professional Markdown report using the research below.

        Topic: {self.state.topic}
        Audience: {self.state.audience}

        Research:
        {self.state.research}

        Include:
        1. Title
        2. Executive summary
        3. Introduction and background
        4. Main findings with clear headings
        5. Analysis and implications
        6. Recommendations or conclusion
        7. References with URLs from the research, when available

        Requirements:
        - Write clearly for the specified audience.
        - Do not fabricate statistics, quotations, or references.
        - Distinguish verified findings from uncertain information.
        - Return the report itself in Markdown.
        """

        self.state.draft = run_crew(
            role="Report Writer",
            goal="Produce a clear, accurate, review-ready report.",
            backstory="You are a professional writer who creates structured, evidence-based reports.",
            task_description=draft_task,
            expected_output="A complete Markdown report ready for human review."
        )

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        draft_path = save_text(
            f"draft_{timestamp}.md",
            self.state.draft
        )

        print(f"Draft saved for review: {draft_path}")

    @listen(write_report)
    def review_draft(self):
        print("\n=== Human Review ===")

        while True:
            print("\n" + "=" * 60)
            print(self.state.draft)
            print("=" * 60)

            choice = input(
                "\nChoose: approve / revise / reject: "
            ).strip().lower()

            if choice in ("approve", "approved", "a"):
                self.state.decision = "approved"
                self.state.status = "approved"
                self.state.feedback = ""
                break

            elif choice in ("reject", "rejected", "r"):
                self.state.decision = "rejected"
                self.state.status = "rejected"
                self.state.feedback = input(
                    "Reason for rejection (optional): "
                ).strip()
                break

            elif choice in ("revise", "revision", "v"):
                if self.state.revision_count >= self.state.max_revisions:
                    print("Maximum revisions reached. Report rejected.")
                    self.state.decision = "rejected"
                    self.state.status = "rejected"
                    self.state.feedback = "Maximum revision limit reached."
                    break

                feedback = input(
                    "Describe the changes required: "
                ).strip()

                if not feedback:
                    print("Please enter revision feedback.")
                    continue

                self.state.feedback = feedback
                self.state.revision_count += 1

                print(
                    f"\nGenerating revision "
                    f"{self.state.revision_count}/"
                    f"{self.state.max_revisions}..."
                )

                revision_task = f"""
                Revise the report using the reviewer's feedback.

                Topic: {self.state.topic}
                Audience: {self.state.audience}

                Original report:
                {self.state.draft}

                Research brief:
                {self.state.research}

                Reviewer feedback:
                {self.state.feedback}

                Preserve accurate content and relevant source URLs.
                Apply the feedback carefully. Do not invent facts or citations.
                Return the complete revised report in Markdown.
                """

                self.state.draft = run_crew(
                    role="Report Editor",
                    goal="Improve the report according to human feedback.",
                    backstory="You are a meticulous editor who preserves factual accuracy.",
                    task_description=revision_task,
                    expected_output="The complete revised Markdown report."
                )

                revision_path = save_text(
                    f"revision_{self.state.revision_count}.md",
                    self.state.draft
                )
                print(f"Revision saved: {revision_path}")

            else:
                print("Invalid choice. Enter approve, revise, or reject.")

        print(f"\nReview decision: {self.state.decision}")

    @router(review_draft)
    def route_decision(self):
        if self.state.decision == "approved":
            return "approved"
        return "rejected"

    @listen("approved")
    def publish_approved_report(self):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        self.state.final_path = save_text(
            f"final_report_{timestamp}.md",
            self.state.draft
        )

        save_json(
            f"approval_record_{timestamp}.json",
            {
                "topic": self.state.topic,
                "audience": self.state.audience,
                "decision": self.state.decision,
                "status": self.state.status,
                "revision_count": self.state.revision_count,
                "final_report_path": self.state.final_path,
                "reviewed_at": datetime.now(timezone.utc).isoformat()
            }
        )

        print("\nREPORT APPROVED")
        print(f"Final report saved to: {self.state.final_path}")

    @listen("rejected")
    def record_rejection(self):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        record_path = save_json(
            f"rejection_record_{timestamp}.json",
            {
                "topic": self.state.topic,
                "audience": self.state.audience,
                "decision": self.state.decision,
                "status": self.state.status,
                "feedback": self.state.feedback,
                "revision_count": self.state.revision_count,
                "final_report_saved": False,
                "reviewed_at": datetime.now(timezone.utc).isoformat()
            }
        )

        print("\nREPORT NOT APPROVED")
        print(f"Review record saved to: {record_path}")
        print("No final report was published.")


print("ApprovalReportFlow class defined successfully.")


ApprovalReportFlow class defined successfully.


In [ ]:

flow = ApprovalReportFlow()

result = flow.kickoff()

print("\n=== FLOW EXECUTION FINISHED ===")
print("Status:", flow.state.status)
print("Decision:", flow.state.decision)
print("Revisions:", flow.state.revision_count)

if flow.state.status == "approved":
    print("Final report:", flow.state.final_path)
else:
    print("No final report was published.")


╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ApprovalReportFlow                                                                                       │
│  ID: f57d05e4-753d-43dd-bb70-a3ed0d6f26df                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ApprovalReportFlow                                                                                       │
│  ID: f57d05e4-753d-43dd-bb70-a3ed0d6f26df                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

=== Approval Report Flow ===


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: collect_topic                                                                                          │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Enter the report topic: Impact of Generative AI on Education
Enter the target audience (e.g., students, managers): College students

Topic: Impact of Generative AI on Education
Audience: College students

=== Researching Topic ===


...

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish_approved_report                                                                                │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: route_decision                                                                                         │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: review_draft                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ApprovalReportFlow                                                                                       │
│  ID: f57d05e4-753d-43dd-bb70-a3ed0d6f26df                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FLOW EXECUTION FINISHED ===
Status: approved
Decision: approved
Revisions: 1
Final report: /content/approval_report_output/final_report_20261022_171800.md


In [ ]:
from pathlib import Path
from google.colab import files

if flow.state.status == "approved" and flow.state.final_path:
    report_path = Path(flow.state.final_path)

    if report_path.exists():
        print("=== FINAL APPROVED REPORT ===\n")
        print(report_path.read_text(encoding="utf-8"))

        files.download(str(report_path))
    else:
        print(f"Report file not found: {report_path}")
else:
    print("No approved Markdown report is available to download.")